In [2]:
import os
from pathlib import Path
from src.utils import load_json
from src.chem_draw import draw_reaction
from IPython.display import SVG
import pandas as pd
import polars as pl
from collections import defaultdict
import ipywidgets as widgets
%load_ext memory_profiler
print(pd.__version__)

2.2.3


In [2]:
assets_dir = Path(os.environ.get("BOTTLE_EXPANSION_ASSETS"))

In [3]:
%%memit
assets = {}
for p in assets_dir.glob('*'):
    if p.suffix == '.json':
        assets[p.stem] = load_json(p)

    else:
        assets['svgs'] = {}
        for s in p.glob('*'):
            assets['svgs'][s.stem] = SVG(s).data

peak memory: 1144.17 MiB, increment: 939.99 MiB


In [ ]:
tmp = defaultdict(list)
for k, v in assets['found_paths'].items():
    for _k, _v in v.items():    
        tmp[_k].append(_v)

df = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

df.to_parquet(
    'asset_storage/paths.parquet',
    index=False,
)

df.head()

In [ ]:
len(df)

In [ ]:
tmp = defaultdict(list)
for k, v in assets['known_reactions'].items():
    for _k, _v in v.items():

        if _k == 'image':
            continue
        
        tmp[_k].append(_v)

df = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

df.to_parquet(
    'asset_storage/known_reactions_no_image.parquet',
    index=False,
)

df.head()

In [ ]:
tmp = defaultdict(list)
for k, v in assets['known_reactions'].items():
    
    # Add the rest
    for _k, _v in v.items():
        if _k == 'image':
            tmp[_k].append(assets['svgs'][_v.removesuffix('.svg')])
        else:
            tmp[_k].append(_v)

krs = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

krs.to_parquet(
    'asset_storage/known_reactions.parquet',
    index=False,
)

krs.head()

In [ ]:
tmp = defaultdict(list)
for k, v in assets['predicted_reactions'].items():
    
    # Add the rest
    for _k, _v in v.items():
        if _k == 'image':
            tmp[_k].append(assets['svgs'][_v.removesuffix('.svg')])
        else:
            tmp[_k].append(_v)

prs = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

all_rxns = pd.concat([krs, prs], ignore_index=True)

all_rxns.to_parquet(
    'asset_storage/reactions.parquet',
    index=False,
)

all_rxns.head()

In [ ]:
tmp = defaultdict(list)
to_enz = defaultdict(list)
for k, v in assets['known_reactions'].items():
    
    # Add the rest
    for _k, _v in v.items():
        if _k == 'image':
            tmp[_k].append(assets['svgs'][_v.removesuffix('.svg')])
        elif _k == 'enzymes':
            tmp[_k].append([e['uniprot_id'] for e in _v])

            for e in _v:
                for ek, ev in e.items():
                    to_enz[ek].append(ev)

        else:
            tmp[_k].append(_v)

krs = pd.DataFrame(
    data=tmp,
    columns=tmp.keys()
)

all_rxns = pd.concat([krs, prs], ignore_index=True)

all_rxns.to_parquet(
    'asset_storage/reactions_no_enzymes.parquet',
    index=False,
)

enzymes = pd.DataFrame(
    data=to_enz,
    columns=to_enz.keys()
)

enzymes.to_parquet(
    'asset_storage/enzymes.parquet',
    index=False,
)

enzymes.head()

In [ ]:
to_enz

In [ ]:
tmp = list(assets['svgs'].items())

df = pd.DataFrame(
    data=tmp,
    columns=['reaction_id', 'image']
)

df.to_parquet(
    'asset_storage/svgs.parquet',
    index=False,
)


df.head()

Loading in

In [17]:
%%memit
lrxns = pd.read_parquet('asset_storage/reactions.parquet')

peak memory: 4916.50 MiB, increment: 3535.63 MiB


In [ ]:
%%memit
lrxns = pd.read_parquet('asset_storage/known_reactions_no_image.parquet')

Draw on the fly vs precompute images

In [ ]:
%%timeit -n 1_00
svgfig = draw_reaction(all_rxns.loc[0, 'smarts'], auto_scl=True)
svg = svgfig.to_str()

In [ ]:
%%timeit -n 1_00
svg = widgets.Image.from_file(assets_dir / 'svgs' / f"{all_rxns.loc[0, 'id']}.svg")

In [ ]:
%%timeit -n 1_00
svg = widgets.Image.from_file(assets_dir / 'svgs' / f"{all_rxns.loc[0, 'id']}.svg")

In [16]:
%%timeit -n 1_00
svg = lrxns.loc[0, 'image']
widgets.HTML(value=svg)

2.82 ms ± 630 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
